# Parcel Fan Chart — Florida & Houston

**Goal:** Retrieve and visualize the full P10–P90 fan chart for individual parcels using the parcel account number.

Coverage: **Florida statewide** (all 67 counties) and **Houston metro** (HCAD).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/homecastr/homecastr-cookbook/blob/main/examples/getting_started/03_parcel_fan_chart.ipynb)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from homecastr import HomecastrClient

client = HomecastrClient(os.getenv("HOMECASTR_API_KEY", "hc_demo_public_readonly"))

## Finding Parcel Account Numbers

- **Florida:** Search at [floridarevenue.com/property](https://floridarevenue.com/property) or your county property appraiser site
- **Houston (HCAD):** Search at [hcad.org/property-search](https://hcad.org/property-search)

In [ ]:
# Florida parcel — confirmed live
FL_ACCT = "00404328020003500"

fan_df = client.forecast.by_parcel.retrieve_fan_chart(FL_ACCT)

print(f"Parcel:        {fan_df.attrs['acct']}")
print(f"Current value: ${fan_df.attrs['current_value']:,}")
print(f"Origin year:   {fan_df.attrs['origin_year']}")
print()
print(fan_df[['year', 'horizon_months', 'p10', 'p25', 'p50', 'p75', 'p90']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.fill_between(fan_df["year"], fan_df["p10"] / 1e3, fan_df["p90"] / 1e3,
                alpha=0.15, color="steelblue", label="P10–P90")
ax.fill_between(fan_df["year"], fan_df["p25"] / 1e3, fan_df["p75"] / 1e3,
                alpha=0.3, color="steelblue", label="P25–P75")
ax.plot(fan_df["year"], fan_df["p50"] / 1e3, color="steelblue", lw=2, label="P50 (median)")
ax.axhline(fan_df.attrs["current_value"] / 1e3, color="gray", ls="--", lw=1, label="Current value")

ax.set_title(f"Homecastr Fan Chart — FL Parcel {FL_ACCT}")
ax.set_xlabel("Year")
ax.set_ylabel("Value ($000s)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.0f}k"))
ax.legend()
plt.tight_layout()
plt.show()

## Bulk Parcel Lookup

In [ ]:
# Mix of Florida parcels — all confirmed to have data
FL_PARCELS = [
    "00404328020003500",
    "00404328070001380",
    "00404328070001990",
    "204623T3036000110",
]

rows = []
for acct in FL_PARCELS:
    try:
        result = client.forecast.by_parcel.retrieve(acct)
        fan = result.get("fan_chart", [])
        h12 = next((f for f in fan if f.get("horizon_months") == 12), fan[1] if len(fan) > 1 else {})
        h60 = next((f for f in fan if f.get("horizon_months") == 60), {})
        rows.append({
            "acct": result.get("acct", acct),
            "current_value": result.get("current_value"),
            "p50_12m": h12.get("p50"),
            "p50_60m": h60.get("p50"),
            "appreciation_12m": (
                round((h12["p50"] / result["current_value"] - 1) * 100, 1)
                if h12.get("p50") and result.get("current_value") else None
            ),
        })
    except Exception as e:
        rows.append({"acct": acct, "error": str(e)})

df = pd.DataFrame(rows)
print(df.to_string(index=False))